# Adult Income — Step 2/3: Modeling & Model Selection

This notebook implements the **Modeling** parts of the project:
- Step 2: Implementation & Data Pipeline (re-usable, leakage-safe preprocessing)
- Step 3: Model Selection & Justification (compare several classifiers, light tuning, pick a final model)

We will evaluate **Logistic Regression, Gaussian Naive Bayes, Decision Tree, SVC (SVM classifier), and KNN**.


In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score, average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# todo check if needed
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

## 1) Data Loading & Deterministic Cleaning (re-using EDA decisions)

We re-implement the exact cleaning choices from EDA so the modeling is **fully reproducible**:
- Replace `?` → `Unknown` on selected categoricals
- Trim whitespace in all object columns
- Normalize income labels to remove trailing dots
- Drop `fnlwgt` and `education` (redundant / undesirable in modeling)


In [ ]:
def load_and_basic_clean(train_path="data/adult.data", test_path="data/adult.test"):
    column_names = [
        "age","workclass","fnlwgt","education","education-num","marital-status",
        "occupation","relationship","race","sex","capital-gain","capital-loss",
        "hours-per-week","native-country","income"
    ]
    df_train = pd.read_csv(train_path, names=column_names)
    df_test  = pd.read_csv(test_path, skiprows=1, names=column_names)

    for col in ["workclass", "occupation", "native-country"]:
        df_train[col] = df_train[col].replace("?", "Unknown")
        df_test[col]  = df_test[col].replace("?", "Unknown")

    for df in (df_train, df_test):
        obj_cols = df.select_dtypes(include="object").columns
        for c in obj_cols:
            df[c] = df[c].str.strip()

    df_train["income"] = df_train["income"].str.replace(".", "", regex=False)
    df_test["income"]  = df_test["income"].str.replace(".", "", regex=False)

    df_train = df_train.drop(columns=["fnlwgt", "education"]) 
    df_test  = df_test.drop(columns=["fnlwgt", "education"]) 

    return df_train, df_test

df_train, df_test = load_and_basic_clean()
print(df_train.head(2))
print("\nTrain shape:", df_train.shape, " Test shape:", df_test.shape)

## 2) Target Encoding & Holdout

- **Target:** `income` → binary (1 for `>50K`, 0 otherwise)
- We reserve the official `adult.test` as the **final holdout** to avoid leakage.


In [ ]:
TARGET = "income"
X = df_train.drop(columns=[TARGET])
y = (df_train[TARGET] == ">50K").astype(int)

X_holdout = df_test.drop(columns=[TARGET])
y_holdout = (df_test[TARGET] == ">50K").astype(int)

print("Positive rate in train:", y.mean().round(3), " | Positive rate in holdout:", y_holdout.mean().round(3))

## 3) Leakage-safe Preprocessing Pipeline

- `SimpleImputer(median)` for numerics (robust to outliers); `most_frequent` for categoricals
- `FunctionTransformer(np.log1p)` on **capital-gain/loss** (heavily right-skewed)
- `StandardScaler` for numeric features
- `OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False)` for categoricals


In [ ]:
numeric_cols_all = X.select_dtypes(include=[np.number]).columns.tolist()
skewed_numeric = [c for c in ["capital-gain", "capital-loss"] if c in numeric_cols_all]
base_numeric   = [c for c in numeric_cols_all if c not in skewed_numeric]
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numeric_base = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

numeric_skewed = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
    StandardScaler()
)

categorical = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False)
)

preprocess = ColumnTransformer(
    transformers=[
        ("num_base",   numeric_base,   base_numeric),
        ("num_skewed", numeric_skewed, skewed_numeric),
        ("cat",        categorical,    categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Base numeric:", base_numeric)
print("Skewed numeric:", skewed_numeric)
print("Categorical:", categorical_cols)

## 4) Models Under Comparison
- Logistic Regression
- Gaussian Naive Bayes
- Decision Tree
- SVC (with probability=True for ROC/PR)
- KNN Classifier


In [ ]:
pipe_logreg = Pipeline(steps=[("preprocess", preprocess),
                             ("clf", LogisticRegression(max_iter=1000))])

pipe_nb     = Pipeline(steps=[("preprocess", preprocess),
                             ("clf", GaussianNB())])

pipe_tree   = Pipeline(steps=[("preprocess", preprocess),
                             ("clf", DecisionTreeClassifier(random_state=42))])

pipe_svc    = Pipeline(steps=[("preprocess", preprocess),
                             ("clf", SVC(probability=True, random_state=42))])

pipe_knn    = Pipeline(steps=[("preprocess", preprocess),
                             ("clf", KNeighborsClassifier())])

models = {
    "LogReg": pipe_logreg,
    "GaussianNB": pipe_nb,
    "DecisionTree": pipe_tree,
    "SVC": pipe_svc,
    "KNN": pipe_knn
}
list(models.keys())

## 5) Baseline Cross-Validation (Stratified 5-fold)
We compute Accuracy, F1, ROC-AUC, and Average Precision (PR AUC) for each model.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'avg_precision': 'average_precision'
}

cv_results = {}
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    summary = {m: (scores[f'test_{m}'].mean(), scores[f'test_{m}'].std()) for m in scoring.keys()}
    cv_results[name] = summary

rows = []
for name, metrics in cv_results.items():
    row = {'Model': name}
    for m, (mean, std) in metrics.items():
        row[m] = f"{mean:.3f} ± {std:.3f}"
    rows.append(row)

cv_table = pd.DataFrame(rows).set_index('Model').sort_values('roc_auc', ascending=False)
cv_table

## 6) Light Hyperparameter Tuning (GridSearchCV)
We lightly tune two strong candidates using ROC-AUC as the objective.


In [ ]:
param_grid_logreg = {
    'clf__C': [0.1, 1.0, 3.0, 10.0],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

param_grid_svc = {
    'clf__C': [0.5, 1.0, 3.0],
    'clf__gamma': ['scale', 0.1, 0.01]
}

gs_logreg = GridSearchCV(models['LogReg'], param_grid_logreg, scoring='roc_auc', cv=cv, n_jobs=-1)
gs_svc    = GridSearchCV(models['SVC'],    param_grid_svc,    scoring='roc_auc', cv=cv, n_jobs=-1)

gs_logreg.fit(X, y)
gs_svc.fit(X, y)

print("Best LogReg:", gs_logreg.best_params_, " | AUC=", round(gs_logreg.best_score_, 3))
print("Best SVC:",    gs_svc.best_params_,    " | AUC=", round(gs_svc.best_score_, 3))

## 7) Final Model → Fit on Full Train, Evaluate Once on Holdout
We select by tuned ROC-AUC, retrain on all training data, and report on the untouched holdout.


In [ ]:
best_estimator = gs_svc.best_estimator_ if gs_svc.best_score_ >= gs_logreg.best_score_ else gs_logreg.best_estimator_
final_name = 'SVC (tuned)' if best_estimator is gs_svc.best_estimator_ else 'LogReg (tuned)'
print("Selected final model:", final_name)

best_estimator.fit(X, y)
y_prob = best_estimator.predict_proba(X_holdout)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("Holdout ROC-AUC:", roc_auc_score(y_holdout, y_prob).round(3))
print("Holdout PR-AUC:", average_precision_score(y_holdout, y_prob).round(3))
print("\nClassification report (0=<=50K, 1=>50K):\n")
print(classification_report(y_holdout, y_pred, digits=3))

ConfusionMatrixDisplay.from_predictions(y_holdout, y_pred)
plt.title(f"Confusion Matrix — {final_name} (Holdout)")
plt.show()

## 8) Justification Summary
- All preprocessing is inside the pipeline (no leakage).
- Compared diverse model families with multiple metrics beyond accuracy.
- Light tuning via GridSearchCV with stratified CV for principled selection.
- Final evaluation reported **once** on the holdout to estimate generalization.
